In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
DATA_PATH = "../data/"

def parse_dates(series):
    return pd.to_datetime(series.astype(str).str.strip('"'), errors="coerce", utc=True)

# Tablas necesarias para el análisis de límites
users = pd.read_csv(DATA_PATH + "users.csv", low_memory=False)
tests = pd.read_csv(DATA_PATH + "tests.csv", low_memory=False)
notes = pd.read_csv(DATA_PATH + "notes.csv", low_memory=False)
cards = pd.read_csv(DATA_PATH + "cards.csv", low_memory=False)
traces = pd.read_csv(DATA_PATH + "traces.csv", low_memory=False, sep=";")
ai_spans = pd.read_csv(DATA_PATH + "ai_spans.csv", low_memory=False)
subs = pd.read_csv(DATA_PATH + "stripe_subscriptions.csv", low_memory=False)
buys = pd.read_csv(DATA_PATH + "buys.csv", low_memory=False)

# Fechas
users["created_at"] = parse_dates(users["created_at"])
users["subscription_created_at"] = parse_dates(users["subscription_created_at"])
tests["created_at"] = parse_dates(tests["created_at"])
notes["created_at"] = parse_dates(notes["created_at"])
cards["created_at"] = parse_dates(cards["created_at"])
traces["created_at"] = parse_dates(traces["created_at"])
subs["created"] = parse_dates(subs["created"])
subs["canceled_at"] = parse_dates(subs["canceled_at"])
buys["created_at"] = parse_dates(buys["created_at"])

if "start_time" in ai_spans.columns:
    ai_spans["start_time"] = parse_dates(ai_spans["start_time"])
if "created_at" in ai_spans.columns:
    ai_spans["created_at"] = parse_dates(ai_spans["created_at"])

print("✅ Datos cargados")
print(f"Usuarios: {len(users):,}")
print(f"AI spans: {len(ai_spans):,}")
print(f"Tests: {len(tests):,}")
print(f"Notes: {len(notes):,}")
print(f"Cards: {len(cards):,}")
print(f"Traces: {len(traces):,}")
print(f"Suscripciones: {len(subs):,}")
print(f"Compras extras: {len(buys):,}")

print(f"\n¿questions cargado? {'questions' in dir()}")

✅ Datos cargados
Usuarios: 32,636
AI spans: 269,383
Tests: 88,995
Notes: 858,019
Cards: 914,534
Traces: 112,353
Suscripciones: 627
Compras extras: 839

¿questions cargado? False


In [3]:
# CAMBIO 5: verificación de fechas
print(f"\nVerificación de fechas:")
print(f"  users.created_at válidos: {users['created_at'].notna().sum():,}")
print(f"  tests.created_at válidos: {tests['created_at'].notna().sum():,}")
print(f"  traces.created_at válidos: {traces['created_at'].notna().sum():,}")


Verificación de fechas:
  users.created_at válidos: 32,636
  tests.created_at válidos: 88,995
  traces.created_at válidos: 112,353


## Límites de uso por plan (mensuales)

| Feature | FREE | MagIA (6,99€) | SuperMagIA (12,99€) |
|:---|:---:|:---:|:---:|
| Preguntas test generadas | 250 | 1.500 | 4.000 |
| Flashcards generadas | 500 | 3.000 | 8.000 |
| Chat budget | 0,05$ | 1,20$ | 2,80$ |
| Chat SuperMagIA | ❌ | 30% budget | 75% budget |

**Los límites son mensuales y se resetean cada mes.**

**Tipos de generación en ai_spans (related_object_type en traces):**
- Generación tests desde apuntes → `test_from_notes`
- Generación tests similares → `test_similar_to_exams`
- Extracción tests de exámenes → `test_extract_from_exams`
- Generación flashcards → `flashcards_new`
- Chat → `chat_conversation`

In [4]:
# ============================================================
# Segmentación de usuarios
# ============================================================
# Crea users_base con la clasificación de cada usuario:
# - es_free_puro: True si nunca ha tenido suscripción real en Stripe
# - segmento: 'free_puro', 'pagador_activo', 'churned'

users_base = users.copy()

# Normalizar stripe_customer_id (vacíos pueden venir como '' o NaN)
users_base['stripe_customer_id_clean'] = (
    users_base['stripe_customer_id']
    .replace('', pd.NA)
    .where(users_base['stripe_customer_id'].notna(), pd.NA)
)

# Free puro: subscription_type=free Y sin stripe_customer_id
users_base['es_free_puro'] = (
    (users_base['subscription_type'] == 'free') &
    (users_base['stripe_customer_id_clean'].isna())
)

# Segmento detallado
def clasificar_segmento(row):
    if row['es_free_puro']:
        return 'free_puro'
    elif row['subscription_type'] != 'free':
        return 'pagador_activo'
    else:
        # tiene stripe pero ahora es free → churned
        return 'churned'

users_base['segmento'] = users_base.apply(clasificar_segmento, axis=1)

# Verificación
print(f"Usuarios totales: {len(users_base):,}")
print(f"Free puros: {users_base['es_free_puro'].sum():,}")
print(f"\nDistribución por segmento:")
print(users_base['segmento'].value_counts())

# Validación: los conteos deben coincidir con el TFG
assert users_base['es_free_puro'].sum() == 32017, "Free puros no coincide"
print(f"\n✅ Segmentación validada")

Usuarios totales: 32,636
Free puros: 32,017

Distribución por segmento:
segmento
free_puro         32017
pagador_activo      516
churned             103
Name: count, dtype: int64

✅ Segmentación validada


In [5]:
# Ver tipos de generación disponibles en traces
print("=== TIPOS DE GENERACIÓN EN TRACES ===")
print(traces['related_object_type'].value_counts())

=== TIPOS DE GENERACIÓN EN TRACES ===
related_object_type
test_from_notes            82809
flashcards_new             11606
chat_conversation           7514
test_extract_from_exams     5985
test_similar_to_exams       3019
flashcards                  1364
from_notes                    28
extract_from_exams            21
similar_to_exams               7
Name: count, dtype: int64


## ⚠️ Nota sobre tipos de generación

Los tipos con pocos registros son probablemente nombres antiguos 
de las mismas funcionalidades:
- `from_notes` (28) → versión antigua de `test_from_notes`
- `extract_from_exams` (21) → versión antigua de `test_extract_from_exams`  
- `similar_to_exams` (7) → versión antigua de `test_similar_to_exams`
- `flashcards` (1.364) → versión antigua de `flashcards_new`

**Para el análisis los agruparemos con sus equivalentes modernos.**

In [6]:
# Unir traces con ai_spans
generaciones = traces.merge(
    ai_spans[['trace_id', 'cost', 'prompt_tokens', 
              'completion_tokens', 'status', 'model_mask']],
    left_on='id',
    right_on='trace_id',
    how='inner'
)

# Filtrar solo llamadas exitosas (status = 'completed')
generaciones = generaciones[generaciones['status'] == 'completed'].copy()

# Agrupar tipos antiguos con sus equivalentes modernos
mapeo_tipos = {
    'from_notes': 'test_from_notes',
    'extract_from_exams': 'test_extract_from_exams',
    'similar_to_exams': 'test_similar_to_exams',
    'flashcards': 'flashcards_new'
}
generaciones['tipo'] = generaciones['related_object_type'].replace(mapeo_tipos)

# Añadir mes para análisis mensual
generaciones['mes'] = generaciones['created_at'].dt.to_period('M')

print(f"Total generaciones exitosas: {len(generaciones):,}")
print(f"\nPor tipo:")
print(generaciones['tipo'].value_counts())

Total generaciones exitosas: 255,588

Por tipo:
tipo
test_from_notes            177374
chat_conversation           35347
flashcards_new              24604
test_extract_from_exams     12026
test_similar_to_exams        6237
Name: count, dtype: int64


C:\Users\patri_ey1tp50\AppData\Local\Temp\ipykernel_18504\3325014565.py:23: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  generaciones['mes'] = generaciones['created_at'].dt.to_period('M')


In [7]:
# Cargar questions
questions = pd.read_csv(DATA_PATH + "questions.csv", low_memory=False)

print(f"Total preguntas: {len(questions):,}")
print(f"Columnas: {list(questions.columns)}")

Total preguntas: 1,888,944
Columnas: ['id', 'content', 'explication', 'ai_assisted', 'is_updated', 'new_version_id', 'is_random', 'test_id', 'options', 'version', 'created_at', 'updated_at', 'deleted_at']


In [8]:
# Verificar que para los pagadores reales, subscription_created_at coincide
# con su primera fecha de cobro en Stripe
import numpy as np

usuarios_con_stripe = users[users['stripe_customer_id'].notna()].copy()

# Cruzar con la primera suscripción de cada cliente en Stripe
subs_primera = subs.sort_values('created').drop_duplicates('customer', keep='first')
usuarios_con_stripe = usuarios_con_stripe.merge(
    subs_primera[['customer', 'created']].rename(columns={'created': 'sub_stripe_real'}),
    left_on='stripe_customer_id', right_on='customer', how='left'
)

# Diferencia entre subscription_created_at del usuario y la fecha real de Stripe
usuarios_con_stripe['diff_horas'] = (
    usuarios_con_stripe['subscription_created_at'] - usuarios_con_stripe['sub_stripe_real']
).dt.total_seconds() / 3600

print(f"Usuarios con stripe_customer_id: {len(usuarios_con_stripe):,}")
print(f"Con suscripción real en stripe_subscriptions: {usuarios_con_stripe['sub_stripe_real'].notna().sum():,}")
print(f"Con subscription_created_at relleno: {usuarios_con_stripe['subscription_created_at'].notna().sum():,}")
print(f"\nDiferencia en horas entre los dos campos (para los que tienen ambos):")
print(usuarios_con_stripe['diff_horas'].describe())

# ¿Cuántos tienen una diferencia mayor a 24 horas?
mas_un_dia = (usuarios_con_stripe['diff_horas'].abs() > 24).sum()
print(f"\nUsuarios con diferencia > 24h entre los dos campos: {mas_un_dia}")

Usuarios con stripe_customer_id: 619
Con suscripción real en stripe_subscriptions: 615
Con subscription_created_at relleno: 619

Diferencia en horas entre los dos campos (para los que tienen ambos):
count     615.000000
mean       10.003520
std       123.419919
min        -0.110371
25%         0.000000
50%         0.000000
75%         0.000000
max      2093.141389
Name: diff_horas, dtype: float64

Usuarios con diferencia > 24h entre los dos campos: 5


In [9]:
# ============================================================
# Preguntas IA: superación del límite free
# Universo: todas las ventanas donde el usuario ESTABA EN ESTADO FREE,
# independientemente de lo que haya hecho después.
# ============================================================

# 1. Filtrar preguntas IA y cruzar con tests
questions["ai_assisted_bool"] = (
    questions["ai_assisted"].astype(str).str.strip().str.lower()
    .isin(["true", "t", "1", "yes", "y"])
)
questions_ai = questions[questions["ai_assisted_bool"] == True].copy()

questions_ai = questions_ai.merge(
    tests[["id", "user_id", "created_at"]].rename(columns={
        "id": "test_id",
        "created_at": "test_created_at"
    }),
    on="test_id", how="left"
)

# 2. Cruzar con users para fecha de registro y fecha de suscripción
questions_ai = questions_ai.merge(
    users[["id", "created_at", "subscription_created_at",
           "stripe_customer_id"]].rename(columns={
        "id": "user_id",
        "created_at": "user_created_at"
    }),
    on="user_id", how="left"
)

# 3. Determinar si la pregunta se generó CUANDO el usuario era free
#    Una pregunta se considera "en estado free" si:
#    (a) el usuario nunca pagó (stripe_customer_id IS NULL), o
#    (b) el usuario pagó después de la fecha de la pregunta
questions_ai["era_free_en_ese_momento"] = (
    questions_ai["stripe_customer_id"].isna() |
    (questions_ai["test_created_at"] < questions_ai["subscription_created_at"])
)

# 4. Quedarse solo con preguntas generadas en estado free
preguntas_estado_free = questions_ai[questions_ai["era_free_en_ese_momento"] == True].copy()

# 5. Calcular días desde registro y ventana de 30 días
preguntas_estado_free["dias_desde_registro"] = (
    preguntas_estado_free["test_created_at"] - preguntas_estado_free["user_created_at"]
).dt.days

preguntas_estado_free = preguntas_estado_free.dropna(subset=["dias_desde_registro"])
preguntas_estado_free = preguntas_estado_free[preguntas_estado_free["dias_desde_registro"] >= 0]
preguntas_estado_free["ventana_30d"] = (preguntas_estado_free["dias_desde_registro"] // 30).astype(int)

# 6. Contar preguntas por usuario y ventana en estado free
preguntas_free_ventana = (
    preguntas_estado_free
    .groupby(["user_id", "ventana_30d"])
    .size()
    .reset_index(name="preguntas_ia")
)

LIMITE_FREE_PREGUNTAS = 250
preguntas_free_ventana["supera_limite"] = (
    preguntas_free_ventana["preguntas_ia"] > LIMITE_FREE_PREGUNTAS
)

# 7. Resultados
ids_que_superaron = set(preguntas_free_ventana.loc[
    preguntas_free_ventana["supera_limite"], "user_id"
])

usuarios_con_actividad_free = preguntas_estado_free["user_id"].nunique()
usuarios_que_superaron = len(ids_que_superaron)

print(f"=== PREGUNTAS - Superación de límite en estado free ===")
print(f"Usuarios con preguntas IA en estado free: {usuarios_con_actividad_free:,}")
print(f"Usuarios que superaron límite alguna vez: {usuarios_que_superaron:,}")
print(f"Porcentaje: {usuarios_que_superaron/usuarios_con_actividad_free*100:.2f}%")

# 8. De los que superaron, ¿cuántos pagaron después?
df_superan = users[users["id"].isin(ids_que_superaron)]
con_stripe = df_superan["stripe_customer_id"].notna().sum()
pagador_activo = (df_superan["subscription_type"] != "free").sum()
print(f"\nDe los que superaron el límite:")
print(f"  Pagaron alguna vez (Stripe): {con_stripe} ({con_stripe/usuarios_que_superaron*100:.1f}%)")
print(f"  Pagador activo ahora: {pagador_activo} ({pagador_activo/usuarios_que_superaron*100:.1f}%)")

=== PREGUNTAS - Superación de límite en estado free ===
Usuarios con preguntas IA en estado free: 17,136
Usuarios que superaron límite alguna vez: 611
Porcentaje: 3.57%

De los que superaron el límite:
  Pagaron alguna vez (Stripe): 27 (4.4%)
  Pagador activo ahora: 17 (2.8%)


In [10]:
notes["ai_assisted_bool"] = (
    notes["ai_assisted"].astype(str).str.strip().str.lower()
    .isin(["true", "t", "1", "yes", "y"])
)
notes_ai = notes[notes["ai_assisted_bool"] == True].copy()

cards_ai = cards.merge(
    notes_ai[["id", "user_id", "created_at"]].rename(columns={
        "id": "note_id", "created_at": "note_created_at"
    }),
    on="note_id", how="inner"
)

cards_ai = cards_ai.merge(
    users[["id", "created_at", "subscription_created_at",
           "stripe_customer_id"]].rename(columns={
        "id": "user_id", "created_at": "user_created_at"
    }),
    on="user_id", how="left"
)

cards_ai["era_free_en_ese_momento"] = (
    cards_ai["stripe_customer_id"].isna() |
    (cards_ai["note_created_at"] < cards_ai["subscription_created_at"])
)

cards_estado_free = cards_ai[cards_ai["era_free_en_ese_momento"] == True].copy()

cards_estado_free["dias_desde_registro"] = (
    cards_estado_free["note_created_at"] - cards_estado_free["user_created_at"]
).dt.days
cards_estado_free = cards_estado_free.dropna(subset=["dias_desde_registro"])
cards_estado_free = cards_estado_free[cards_estado_free["dias_desde_registro"] >= 0]
cards_estado_free["ventana_30d"] = (cards_estado_free["dias_desde_registro"] // 30).astype(int)

flashcards_free_ventana = (
    cards_estado_free.groupby(["user_id", "ventana_30d"])
    .size().reset_index(name="cards_generadas")
)

LIMITE_FREE_FLASHCARDS = 500
flashcards_free_ventana["supera_limite"] = (
    flashcards_free_ventana["cards_generadas"] > LIMITE_FREE_FLASHCARDS
)

ids_superan_f = set(flashcards_free_ventana.loc[
    flashcards_free_ventana["supera_limite"], "user_id"
])
n_total = cards_estado_free["user_id"].nunique()
n_superan = len(ids_superan_f)

print(f"=== FLASHCARDS - Superación de límite en estado free ===")
print(f"Usuarios con flashcards en estado free: {n_total:,}")
print(f"Usuarios que superaron límite alguna vez: {n_superan:,}")
print(f"Porcentaje: {n_superan/n_total*100:.2f}%")

df_superan = users[users["id"].isin(ids_superan_f)]
con_stripe = df_superan["stripe_customer_id"].notna().sum()
pagador_activo = (df_superan["subscription_type"] != "free").sum()
print(f"\nDe los que superaron el límite:")
print(f"  Pagaron alguna vez (Stripe): {con_stripe} ({con_stripe/n_superan*100:.1f}%)")
print(f"  Pagador activo ahora: {pagador_activo} ({pagador_activo/n_superan*100:.1f}%)")

=== FLASHCARDS - Superación de límite en estado free ===
Usuarios con flashcards en estado free: 5,599
Usuarios que superaron límite alguna vez: 201
Porcentaje: 3.59%

De los que superaron el límite:
  Pagaron alguna vez (Stripe): 10 (5.0%)
  Pagador activo ahora: 10 (5.0%)


In [11]:
# Repetir lógica para chat
chat_traces = traces[traces["related_object_type"] == "chat_conversation"][
    ["id", "user_id", "created_at"]
].rename(columns={"id": "trace_id", "created_at": "chat_created_at"})

chat_eventos = chat_traces.merge(
    ai_spans[["trace_id", "cost", "status"]],
    on="trace_id", how="inner"
)
chat_eventos = chat_eventos[chat_eventos["status"] == "completed"].copy()

chat_eventos = chat_eventos.merge(
    users[["id", "created_at", "subscription_created_at",
           "stripe_customer_id"]].rename(columns={
        "id": "user_id", "created_at": "user_created_at"
    }),
    on="user_id", how="left"
)

chat_eventos["era_free_en_ese_momento"] = (
    chat_eventos["stripe_customer_id"].isna() |
    (chat_eventos["chat_created_at"] < chat_eventos["subscription_created_at"])
)

chat_estado_free = chat_eventos[chat_eventos["era_free_en_ese_momento"] == True].copy()

chat_estado_free["dias_desde_registro"] = (
    chat_estado_free["chat_created_at"] - chat_estado_free["user_created_at"]
).dt.days
chat_estado_free = chat_estado_free.dropna(subset=["dias_desde_registro"])
chat_estado_free = chat_estado_free[chat_estado_free["dias_desde_registro"] >= 0]
chat_estado_free["ventana_30d"] = (chat_estado_free["dias_desde_registro"] // 30).astype(int)

chat_free_ventana = (
    chat_estado_free.groupby(["user_id", "ventana_30d"])
    .agg(coste_chat=("cost", "sum"), n_mensajes=("trace_id", "count"))
    .reset_index()
)

LIMITE_FREE_CHAT_USD = 0.05
chat_free_ventana["supera_limite"] = (
    chat_free_ventana["coste_chat"] > LIMITE_FREE_CHAT_USD
)

ids_superan_chat = set(chat_free_ventana.loc[
    chat_free_ventana["supera_limite"], "user_id"
])
n_total = chat_estado_free["user_id"].nunique()
n_superan = len(ids_superan_chat)

print(f"=== CHAT - Superación de presupuesto en estado free ===")
print(f"Usuarios con uso de chat en estado free: {n_total:,}")
print(f"Usuarios que superaron presupuesto alguna vez: {n_superan:,}")
print(f"Porcentaje: {n_superan/n_total*100:.2f}%")

df_superan = users[users["id"].isin(ids_superan_chat)]
con_stripe = df_superan["stripe_customer_id"].notna().sum()
pagador_activo = (df_superan["subscription_type"] != "free").sum()
print(f"\nDe los que superaron el presupuesto:")
print(f"  Pagaron alguna vez (Stripe): {con_stripe} ({con_stripe/n_superan*100:.1f}%)")
print(f"  Pagador activo ahora: {pagador_activo} ({pagador_activo/n_superan*100:.1f}%)")

=== CHAT - Superación de presupuesto en estado free ===
Usuarios con uso de chat en estado free: 2,410
Usuarios que superaron presupuesto alguna vez: 82
Porcentaje: 3.40%

De los que superaron el presupuesto:
  Pagaron alguna vez (Stripe): 19 (23.2%)
  Pagador activo ahora: 17 (20.7%)


In [12]:
# ============================================================
# MAGIA - Superación de límites del plan MagIA
# ============================================================
# Universo: usuarios en estado MagIA (pagadores con plan MagIA)
# Ventanas: desde la fecha de suscripción del usuario
# Límites: 1.500 preguntas, 3.000 flashcards, 1,20$ chat

# Identificar usuarios MagIA: tienen stripe Y su suscripción es de 6,99€ (699) o trimestral magia (1699)
subs_primera = subs.sort_values('created').drop_duplicates('customer', keep='first')

users_magia = users.merge(
    subs_primera[['customer', 'created', 'price_amount']].rename(columns={
        'created': 'fecha_suscripcion_real',
        'price_amount': 'precio_suscripcion'
    }),
    left_on='stripe_customer_id', right_on='customer', how='inner'
)

# Filtrar planes MagIA (precios 699 mensual, 1699 trimestral)
users_magia = users_magia[users_magia['precio_suscripcion'].isin([699, 1699])].copy()
print(f"Usuarios con plan MagIA en algún momento: {len(users_magia):,}")

Usuarios con plan MagIA en algún momento: 463


In [13]:
# ============================================================
# MAGIA - Superación del límite de preguntas (1.500/30 días)
# ============================================================

# Cruzar questions_ai con users_magia
questions_magia = questions_ai.merge(
    users_magia[['id', 'fecha_suscripcion_real']].rename(columns={'id': 'user_id'}),
    on='user_id', how='inner'
)

# Solo preguntas posteriores a la suscripción (en estado MagIA)
questions_magia = questions_magia[
    questions_magia['test_created_at'] >= questions_magia['fecha_suscripcion_real']
].copy()

# Días desde suscripción y ventana de 30 días
questions_magia['dias_desde_sub'] = (
    questions_magia['test_created_at'] - questions_magia['fecha_suscripcion_real']
).dt.days
questions_magia = questions_magia.dropna(subset=['dias_desde_sub'])
questions_magia = questions_magia[questions_magia['dias_desde_sub'] >= 0]
questions_magia['ventana_30d'] = (questions_magia['dias_desde_sub'] // 30).astype(int)

# Contar preguntas por usuario y ventana
preguntas_magia_ventana = (
    questions_magia.groupby(['user_id', 'ventana_30d'])
    .size().reset_index(name='preguntas_ia')
)

LIMITE_MAGIA_PREGUNTAS = 1500
preguntas_magia_ventana['supera_limite'] = (
    preguntas_magia_ventana['preguntas_ia'] > LIMITE_MAGIA_PREGUNTAS
)

n_total = questions_magia['user_id'].nunique()
n_superan = preguntas_magia_ventana.loc[
    preguntas_magia_ventana['supera_limite'], 'user_id'
].nunique()

print(f"=== PREGUNTAS - Superación de límite plan MagIA ===")
print(f"Usuarios MagIA con preguntas IA: {n_total:,}")
print(f"Usuarios que superaron límite ({LIMITE_MAGIA_PREGUNTAS}) alguna vez: {n_superan:,}")
print(f"Porcentaje: {n_superan/n_total*100:.2f}%" if n_total > 0 else "Sin datos")

if n_superan > 0:
    print(f"\nConsumo en las ventanas que superan:")
    print(preguntas_magia_ventana.loc[
        preguntas_magia_ventana['supera_limite'], 'preguntas_ia'
    ].describe().round(0))

=== PREGUNTAS - Superación de límite plan MagIA ===
Usuarios MagIA con preguntas IA: 416
Usuarios que superaron límite (1500) alguna vez: 9
Porcentaje: 2.16%

Consumo en las ventanas que superan:
count       9.0
mean     1630.0
std       107.0
min      1510.0
25%      1590.0
50%      1615.0
75%      1635.0
max      1893.0
Name: preguntas_ia, dtype: float64


In [14]:
# ============================================================
# MAGIA - Superación del límite de flashcards (3.000/30 días)
# ============================================================

cards_magia = cards_ai.merge(
    users_magia[['id', 'fecha_suscripcion_real']].rename(columns={'id': 'user_id'}),
    on='user_id', how='inner'
)

# Solo cards posteriores a la suscripción
cards_magia = cards_magia[
    cards_magia['note_created_at'] >= cards_magia['fecha_suscripcion_real']
].copy()

cards_magia['dias_desde_sub'] = (
    cards_magia['note_created_at'] - cards_magia['fecha_suscripcion_real']
).dt.days
cards_magia = cards_magia.dropna(subset=['dias_desde_sub'])
cards_magia = cards_magia[cards_magia['dias_desde_sub'] >= 0]
cards_magia['ventana_30d'] = (cards_magia['dias_desde_sub'] // 30).astype(int)

flashcards_magia_ventana = (
    cards_magia.groupby(['user_id', 'ventana_30d'])
    .size().reset_index(name='cards_generadas')
)

LIMITE_MAGIA_FLASHCARDS = 3000
flashcards_magia_ventana['supera_limite'] = (
    flashcards_magia_ventana['cards_generadas'] > LIMITE_MAGIA_FLASHCARDS
)

n_total = cards_magia['user_id'].nunique()
n_superan = flashcards_magia_ventana.loc[
    flashcards_magia_ventana['supera_limite'], 'user_id'
].nunique()

print(f"=== FLASHCARDS - Superación de límite plan MagIA ===")
print(f"Usuarios MagIA con flashcards IA: {n_total:,}")
print(f"Usuarios que superaron límite ({LIMITE_MAGIA_FLASHCARDS}) alguna vez: {n_superan:,}")
print(f"Porcentaje: {n_superan/n_total*100:.2f}%" if n_total > 0 else "Sin datos")

if n_superan > 0:
    print(f"\nConsumo en las ventanas que superan:")
    print(flashcards_magia_ventana.loc[
        flashcards_magia_ventana['supera_limite'], 'cards_generadas'
    ].describe().round(0))

=== FLASHCARDS - Superación de límite plan MagIA ===
Usuarios MagIA con flashcards IA: 218
Usuarios que superaron límite (3000) alguna vez: 1
Porcentaje: 0.46%

Consumo en las ventanas que superan:
count       1.0
mean     3227.0
std         NaN
min      3227.0
25%      3227.0
50%      3227.0
75%      3227.0
max      3227.0
Name: cards_generadas, dtype: float64


In [15]:
# ============================================================
# MAGIA - Superación del presupuesto de chat (1,20$/30 días)
# ============================================================

chat_magia = chat_eventos.merge(
    users_magia[['id', 'fecha_suscripcion_real']].rename(columns={'id': 'user_id'}),
    on='user_id', how='inner'
)

# Solo chat posterior a la suscripción
chat_magia = chat_magia[
    chat_magia['chat_created_at'] >= chat_magia['fecha_suscripcion_real']
].copy()

chat_magia['dias_desde_sub'] = (
    chat_magia['chat_created_at'] - chat_magia['fecha_suscripcion_real']
).dt.days
chat_magia = chat_magia.dropna(subset=['dias_desde_sub'])
chat_magia = chat_magia[chat_magia['dias_desde_sub'] >= 0]
chat_magia['ventana_30d'] = (chat_magia['dias_desde_sub'] // 30).astype(int)

chat_magia_ventana = (
    chat_magia.groupby(['user_id', 'ventana_30d'])
    .agg(coste_chat=('cost', 'sum'), n_mensajes=('trace_id', 'count'))
    .reset_index()
)

LIMITE_MAGIA_CHAT_USD = 1.20
chat_magia_ventana['supera_limite'] = (
    chat_magia_ventana['coste_chat'] > LIMITE_MAGIA_CHAT_USD
)

n_total = chat_magia['user_id'].nunique()
n_superan = chat_magia_ventana.loc[
    chat_magia_ventana['supera_limite'], 'user_id'
].nunique()

chat_magia_superan = chat_magia_ventana[
    chat_magia_ventana["supera_limite"] == True
].copy()

n_usuarios_magia_chat = chat_magia_ventana["user_id"].nunique()
n_usuarios_magia_superan = chat_magia_superan["user_id"].nunique()
n_ventanas_magia_superan = len(chat_magia_superan)

print("=== CHAT - Superación de presupuesto plan MagIA ===")
print(f"Usuarios MagIA con uso de chat: {n_usuarios_magia_chat}")
print(f"Usuarios que superaron presupuesto (1.2$) alguna vez: {n_usuarios_magia_superan}")
print(f"Ventanas de 30 días que superaron presupuesto: {n_ventanas_magia_superan}")
print(f"Porcentaje de usuarios que superan: {n_usuarios_magia_superan / n_usuarios_magia_chat * 100:.2f}%")

if n_ventanas_magia_superan > 0:
    display(
        chat_magia_superan[
            ["user_id", "ventana_30d", "coste_chat", "n_mensajes", "supera_limite"]
        ]
    )

=== CHAT - Superación de presupuesto plan MagIA ===
Usuarios MagIA con uso de chat: 190
Usuarios que superaron presupuesto (1.2$) alguna vez: 1
Ventanas de 30 días que superaron presupuesto: 2
Porcentaje de usuarios que superan: 0.53%


,user_id,ventana_30d,coste_chat,n_mensajes,supera_limite
7,019a02ad-5f70-7cbb-94bc-7e605480546c,0,2.022113,498,True
8,019a02ad-5f70-7cbb-94bc-7e605480546c,1,1.228884,157,True


In [16]:
# ============================================================
# SUPERMAGIA - Superación del límite de preguntas (4.000/30 días)
# ============================================================

# Identificar usuarios SuperMagIA (precios 1.299 mensual, 3.199 trimestral)
users_super = users.merge(
    subs_primera[['customer', 'created', 'price_amount']].rename(columns={
        'created': 'fecha_suscripcion_real',
        'price_amount': 'precio_suscripcion'
    }),
    left_on='stripe_customer_id', right_on='customer', how='inner'
)
users_super = users_super[users_super['precio_suscripcion'].isin([1299, 3199])].copy()
print(f"Usuarios con plan SuperMagIA en algún momento: {len(users_super):,}")

# Cruzar con questions_ai
questions_super = questions_ai.merge(
    users_super[['id', 'fecha_suscripcion_real']].rename(columns={'id': 'user_id'}),
    on='user_id', how='inner'
)
questions_super = questions_super[
    questions_super['test_created_at'] >= questions_super['fecha_suscripcion_real']
].copy()

questions_super['dias_desde_sub'] = (
    questions_super['test_created_at'] - questions_super['fecha_suscripcion_real']
).dt.days
questions_super = questions_super.dropna(subset=['dias_desde_sub'])
questions_super = questions_super[questions_super['dias_desde_sub'] >= 0]
questions_super['ventana_30d'] = (questions_super['dias_desde_sub'] // 30).astype(int)

preguntas_super_ventana = (
    questions_super.groupby(['user_id', 'ventana_30d'])
    .size().reset_index(name='preguntas_ia')
)

LIMITE_SUPER_PREGUNTAS = 4000
preguntas_super_ventana['supera_limite'] = (
    preguntas_super_ventana['preguntas_ia'] > LIMITE_SUPER_PREGUNTAS
)

n_total = questions_super['user_id'].nunique()
n_superan = preguntas_super_ventana.loc[
    preguntas_super_ventana['supera_limite'], 'user_id'
].nunique()

print(f"\n=== PREGUNTAS - Superación de límite plan SuperMagIA ===")
print(f"Usuarios SuperMagIA con preguntas IA: {n_total:,}")
print(f"Usuarios que superaron límite ({LIMITE_SUPER_PREGUNTAS}) alguna vez: {n_superan:,}")
if n_total > 0:
    print(f"Porcentaje: {n_superan/n_total*100:.2f}%")

if n_superan > 0:
    print(f"\nConsumo en las ventanas que superan:")
    print(preguntas_super_ventana.loc[
        preguntas_super_ventana['supera_limite'], 'preguntas_ia'
    ].describe().round(0))

Usuarios con plan SuperMagIA en algún momento: 152

=== PREGUNTAS - Superación de límite plan SuperMagIA ===
Usuarios SuperMagIA con preguntas IA: 137
Usuarios que superaron límite (4000) alguna vez: 0
Porcentaje: 0.00%


In [17]:
# ============================================================
# SUPERMAGIA - Superación del límite de flashcards (8.000/30 días)
# ============================================================

cards_super = cards_ai.merge(
    users_super[['id', 'fecha_suscripcion_real']].rename(columns={'id': 'user_id'}),
    on='user_id', how='inner'
)
cards_super = cards_super[
    cards_super['note_created_at'] >= cards_super['fecha_suscripcion_real']
].copy()

cards_super['dias_desde_sub'] = (
    cards_super['note_created_at'] - cards_super['fecha_suscripcion_real']
).dt.days
cards_super = cards_super.dropna(subset=['dias_desde_sub'])
cards_super = cards_super[cards_super['dias_desde_sub'] >= 0]
cards_super['ventana_30d'] = (cards_super['dias_desde_sub'] // 30).astype(int)

flashcards_super_ventana = (
    cards_super.groupby(['user_id', 'ventana_30d'])
    .size().reset_index(name='cards_generadas')
)

LIMITE_SUPER_FLASHCARDS = 8000
flashcards_super_ventana['supera_limite'] = (
    flashcards_super_ventana['cards_generadas'] > LIMITE_SUPER_FLASHCARDS
)

n_total = cards_super['user_id'].nunique()
n_superan = flashcards_super_ventana.loc[
    flashcards_super_ventana['supera_limite'], 'user_id'
].nunique()

print(f"=== FLASHCARDS - Superación de límite plan SuperMagIA ===")
print(f"Usuarios SuperMagIA con flashcards IA: {n_total:,}")
print(f"Usuarios que superaron límite ({LIMITE_SUPER_FLASHCARDS}) alguna vez: {n_superan:,}")
if n_total > 0:
    print(f"Porcentaje: {n_superan/n_total*100:.2f}%")

if n_superan > 0:
    print(f"\nConsumo en las ventanas que superan:")
    print(flashcards_super_ventana.loc[
        flashcards_super_ventana['supera_limite'], 'cards_generadas'
    ].describe().round(0))

=== FLASHCARDS - Superación de límite plan SuperMagIA ===
Usuarios SuperMagIA con flashcards IA: 87
Usuarios que superaron límite (8000) alguna vez: 0
Porcentaje: 0.00%


In [18]:
# ============================================================
# SUPERMAGIA - Superación del presupuesto de chat (2,80$/30 días)
# ============================================================

chat_super = chat_eventos.merge(
    users_super[['id', 'fecha_suscripcion_real']].rename(columns={'id': 'user_id'}),
    on='user_id', how='inner'
)
chat_super = chat_super[
    chat_super['chat_created_at'] >= chat_super['fecha_suscripcion_real']
].copy()

chat_super['dias_desde_sub'] = (
    chat_super['chat_created_at'] - chat_super['fecha_suscripcion_real']
).dt.days
chat_super = chat_super.dropna(subset=['dias_desde_sub'])
chat_super = chat_super[chat_super['dias_desde_sub'] >= 0]
chat_super['ventana_30d'] = (chat_super['dias_desde_sub'] // 30).astype(int)

chat_super_ventana = (
    chat_super.groupby(['user_id', 'ventana_30d'])
    .agg(coste_chat=('cost', 'sum'), n_mensajes=('trace_id', 'count'))
    .reset_index()
)

LIMITE_SUPER_CHAT_USD = 2.80
chat_super_ventana['supera_limite'] = (
    chat_super_ventana['coste_chat'] > LIMITE_SUPER_CHAT_USD
)

n_total = chat_super['user_id'].nunique()
n_superan = chat_super_ventana.loc[
    chat_super_ventana['supera_limite'], 'user_id'
].nunique()

print(f"=== CHAT - Superación de presupuesto plan SuperMagIA ===")
print(f"Usuarios SuperMagIA con uso de chat: {n_total:,}")
print(f"Usuarios que superaron presupuesto ({LIMITE_SUPER_CHAT_USD}$) alguna vez: {n_superan:,}")
if n_total > 0:
    print(f"Porcentaje: {n_superan/n_total*100:.2f}%")

if n_superan > 0:
    print(f"\nConsumo en las ventanas que superan (en $):")
    print(chat_super_ventana.loc[
        chat_super_ventana['supera_limite'], 'coste_chat'
    ].describe().round(3))

=== CHAT - Superación de presupuesto plan SuperMagIA ===
Usuarios SuperMagIA con uso de chat: 96
Usuarios que superaron presupuesto (2.8$) alguna vez: 0
Porcentaje: 0.00%


In [19]:
# ============================================================
# Conversión MagIA → SuperMagIA tras superar techo
# ============================================================

# Usuarios MagIA que superaron alguna vez su techo de preguntas
ids_magia_superan_p = set(preguntas_magia_ventana.loc[
    preguntas_magia_ventana["supera_limite"], "user_id"
])

# Para cada uno: ¿tuvo después una suscripción de SuperMagIA?
# Hay que mirar TODAS sus suscripciones en stripe, no solo la primera
print(f"Usuarios MagIA que tocaron techo: {len(ids_magia_superan_p)}")

# Cruzar con TODAS las suscripciones (no solo la primera)
todas_subs_users = users[users['id'].isin(ids_magia_superan_p)][
    ['id', 'stripe_customer_id']
].merge(
    subs[['customer', 'created', 'price_amount']],
    left_on='stripe_customer_id', right_on='customer', how='left'
)

print(f"\nSuscripciones registradas para esos usuarios:")
print(todas_subs_users.groupby('id').agg(
    n_suscripciones=('price_amount', 'count'),
    precios=('price_amount', lambda x: list(x.dropna()))
))

# ¿Alguno tiene una suscripción de SuperMagIA (1299 o 3199)?
con_super = todas_subs_users[
    todas_subs_users['price_amount'].isin([1299, 3199])
]['id'].unique()

print(f"\nDe los {len(ids_magia_superan_p)} MagIA que tocaron techo de preguntas:")
print(f"  Cuántos tienen alguna suscripción SuperMagIA registrada: {len(con_super)}")

Usuarios MagIA que tocaron techo: 9

Suscripciones registradas para esos usuarios:
                                      n_suscripciones precios
id                                                           
019b51ef-faf6-767e-be37-7d41ec9a1bf0                1   [699]
019b6f66-583e-7804-b898-e2250e2eb343                1  [1699]
019b89d4-6925-7800-b8c1-6e36b81876bf                1   [699]
019b98e0-d5fc-717a-8efa-d4c0c3a91bbe                1   [699]
019b99af-1008-720b-8d5c-eb5aebf38e28                1   [699]
019bace1-3fdb-7266-a38d-830df63ec3ac                1   [699]
019bbf20-7164-7457-a9b7-cfa3f60c4ea6                1   [699]
019bc25d-f008-75b8-bfab-6a592e99c48a                1   [699]
019bcb5e-e730-7290-ab61-ba09a41f9235                1  [1699]

De los 9 MagIA que tocaron techo de preguntas:
  Cuántos tienen alguna suscripción SuperMagIA registrada: 0


In [20]:
print("Modelos reales en ai_spans:")
print(ai_spans['model'].value_counts())

Modelos reales en ai_spans:
model
gemini-2.5-flash-lite    143072
gemini-2.0-flash-lite    109120
gemini-2.5-flash          17175
gemini-2.0-flash             16
Name: count, dtype: int64


### Criterio de identificación de heavy users

Para analizar el solapamiento de uso entre funcionalidades, se construye una tabla agregada a nivel usuario que incluye a todos los usuarios con alguna actividad de IA en preguntas, flashcards o chat. Los usuarios que no han utilizado una funcionalidad concreta reciben valor 0 en dicha variable. Por tanto, los percentiles de uso se calculan sobre el conjunto global de usuarios con actividad IA, no exclusivamente sobre los usuarios activos de cada funcionalidad.

Este criterio permite identificar qué usuarios se sitúan en la cola alta de consumo dentro de la base global de usuarios IA y analizar si el uso intensivo se concentra en una sola funcionalidad o se distribuye entre varias.

In [21]:
# ============================================================
# Análisis de solapamiento de uso entre features
# ============================================================

# Para cada usuario, calcular su uso total en cada feature
# (sin importar plan, durante todo el periodo de tracking)

# 1. Preguntas IA por usuario
preg_x_user = (
    questions_ai.dropna(subset=['user_id'])
    .groupby('user_id').size()
    .reset_index(name='n_preguntas')
)

# 2. Flashcards (cards IA) por usuario
cards_x_user = (
    cards_ai.dropna(subset=['user_id'])
    .groupby('user_id').size()
    .reset_index(name='n_cards')
)

# 3. Mensajes de chat por usuario
chat_x_user = (
    chat_eventos.dropna(subset=['user_id'])
    .groupby('user_id')
    .agg(n_chat=('trace_id', 'count'))
    .reset_index()
)

# Cruzar todo en una tabla de uso por usuario
uso = (
    preg_x_user
    .merge(cards_x_user, on='user_id', how='outer')
    .merge(chat_x_user, on='user_id', how='outer')
    .fillna(0)
)

print(f"Usuarios con alguna actividad de IA: {len(uso):,}")
print(f"\nEstadísticas de uso por feature:")
print(uso[['n_preguntas', 'n_cards', 'n_chat']].describe().round(1))

# Correlación entre las tres features
print(f"\n=== CORRELACIÓN DE USO ENTRE FEATURES ===")
print(uso[['n_preguntas', 'n_cards', 'n_chat']].corr(method='spearman').round(3))

# Identificar heavy users de cada feature (P90)
p90_preg = uso['n_preguntas'].quantile(0.90)
p90_cards = uso['n_cards'].quantile(0.90)
p90_chat = uso['n_chat'].quantile(0.90)

uso['heavy_preg'] = uso['n_preguntas'] > p90_preg
uso['heavy_cards'] = uso['n_cards'] > p90_cards
uso['heavy_chat'] = uso['n_chat'] > p90_chat
uso['n_features_heavy'] = (
    uso['heavy_preg'].astype(int) +
    uso['heavy_cards'].astype(int) +
    uso['heavy_chat'].astype(int)
)

print(f"\n=== HEAVY USERS (>P90) POR Nº DE FEATURES ===")
print(f"P90 preguntas: {p90_preg:.0f}")
print(f"P90 cards: {p90_cards:.0f}")
print(f"P90 chat: {p90_chat:.0f}")
print(f"\nUsuarios heavy en N features distintas:")
print(uso['n_features_heavy'].value_counts().sort_index())

# De los que son heavy en al menos una feature, ¿qué proporción lo es en una sola?
heavy_total = uso[uso['n_features_heavy'] >= 1]
print(f"\nDe los {len(heavy_total):,} heavy users (al menos una feature):")
print(f"  Solo en 1 feature: {(heavy_total['n_features_heavy']==1).sum():,} "
      f"({(heavy_total['n_features_heavy']==1).sum()/len(heavy_total)*100:.1f}%)")
print(f"  En 2 features: {(heavy_total['n_features_heavy']==2).sum():,} "
      f"({(heavy_total['n_features_heavy']==2).sum()/len(heavy_total)*100:.1f}%)")
print(f"  En las 3 features: {(heavy_total['n_features_heavy']==3).sum():,} "
      f"({(heavy_total['n_features_heavy']==3).sum()/len(heavy_total)*100:.1f}%)")

Usuarios con alguna actividad de IA: 20,111

Estadísticas de uso por feature:
       n_preguntas  n_cards   n_chat
count      20111.0  20111.0  20111.0
mean          93.6     42.5      1.8
std          202.3    109.9     15.0
min            0.0      0.0      0.0
25%           20.0      0.0      0.0
50%           40.0      0.0      0.0
75%          120.0     39.0      0.0
max        14793.0   3337.0    738.0

=== CORRELACIÓN DE USO ENTRE FEATURES ===
             n_preguntas  n_cards  n_chat
n_preguntas        1.000    0.014  -0.096
n_cards            0.014    1.000   0.028
n_chat            -0.096    0.028   1.000

=== HEAVY USERS (>P90) POR Nº DE FEATURES ===
P90 preguntas: 242
P90 cards: 150
P90 chat: 2

Usuarios heavy en N features distintas:
n_features_heavy
0    15587
1     3856
2      558
3      110
Name: count, dtype: int64

De los 4,524 heavy users (al menos una feature):
  Solo en 1 feature: 3,856 (85.2%)
  En 2 features: 558 (12.3%)
  En las 3 features: 110 (2.4%)


## ⚠️ Limitación importante - Análisis de flashcards

Las flashcards se lanzaron el 3 de enero de 2026. Los datos disponibles
cubren aproximadamente 5 semanas (enero - febrero 2026), lo que limita
significativamente las conclusiones que pueden extraerse de este análisis.

El número de usuarios que ha podido alcanzar el límite de 500 cards/mes
es necesariamente bajo dado el corto período de disponibilidad de la feature.
Los resultados deben interpretarse como indicativos y no como patrones
consolidados de comportamiento.

In [22]:
# Identificar heavy users de cada feature (P95)
p95_preg = uso['n_preguntas'].quantile(0.95)
p95_cards = uso['n_cards'].quantile(0.95)
p95_chat = uso['n_chat'].quantile(0.95)

uso['heavy_preg'] = uso['n_preguntas'] > p95_preg
uso['heavy_cards'] = uso['n_cards'] > p95_cards
uso['heavy_chat'] = uso['n_chat'] > p95_chat
uso['n_features_heavy'] = (
    uso['heavy_preg'].astype(int) +
    uso['heavy_cards'].astype(int) +
    uso['heavy_chat'].astype(int)
)

print(f"\n=== HEAVY USERS (>P95) POR Nº DE FEATURES ===")
print(f"P95 preguntas: {p95_preg:.0f}")
print(f"P95 cards: {p95_cards:.0f}")
print(f"P95 chat: {p95_chat:.0f}")


=== HEAVY USERS (>P95) POR Nº DE FEATURES ===
P95 preguntas: 250
P95 cards: 217
P95 chat: 6


In [23]:
# Identificar heavy users de cada feature (P99)
p99_preg = uso['n_preguntas'].quantile(0.99)
p99_cards = uso['n_cards'].quantile(0.99)
p99_chat = uso['n_chat'].quantile(0.99)

uso['heavy_preg'] = uso['n_preguntas'] > p99_preg
uso['heavy_cards'] = uso['n_cards'] > p99_cards
uso['heavy_chat'] = uso['n_chat'] > p99_chat
uso['n_features_heavy'] = (
    uso['heavy_preg'].astype(int) +
    uso['heavy_cards'].astype(int) +
    uso['heavy_chat'].astype(int)
)

print(f"\n=== HEAVY USERS (>P99) POR Nº DE FEATURES ===")
print(f"P99 preguntas: {p99_preg:.0f}")
print(f"P99 cards: {p99_cards:.0f}")
print(f"P99 chat: {p99_chat:.0f}")


=== HEAVY USERS (>P99) POR Nº DE FEATURES ===
P99 preguntas: 682
P99 cards: 525
P99 chat: 30


In [24]:
# Identificar heavy users de cada feature (P99.8)
p998_preg = uso['n_preguntas'].quantile(0.998)
p998_cards = uso['n_cards'].quantile(0.998)
p998_chat = uso['n_chat'].quantile(0.998)

uso['heavy_preg'] = uso['n_preguntas'] > p998_preg
uso['heavy_cards'] = uso['n_cards'] > p998_cards
uso['heavy_chat'] = uso['n_chat'] > p998_chat
uso['n_features_heavy'] = (
    uso['heavy_preg'].astype(int) +
    uso['heavy_cards'].astype(int) +
    uso['heavy_chat'].astype(int)
)

print(f"\n=== HEAVY USERS (>P99.8) POR Nº DE FEATURES ===")
print(f"P99.8 preguntas: {p998_preg:.0f}")
print(f"P99.8 cards: {p998_cards:.0f}")
print(f"P99.8 chat: {p998_chat:.0f}")


=== HEAVY USERS (>P99.8) POR Nº DE FEATURES ===
P99.8 preguntas: 1600
P99.8 cards: 709
P99.8 chat: 119


## Resumen metodológico del notebook

**Objetivo del notebook:**  
Analizar si los límites de uso de los planes de PROXUS generan fricción efectiva sobre los usuarios y si dicha fricción se relaciona con monetización o necesidad de cambio de plan. El notebook estudia tres funcionalidades basadas en IA: generación de preguntas de test, generación de flashcards y chat IA. Para cada una se analiza la superación de límites en ventanas de 30 días, tanto en estado gratuito como en los planes MagIA y SuperMagIA.

**Población o muestra analizada:**  
La población general de referencia está formada por 32.636 usuarios activos de PROXUS a fecha de corte 7 de febrero de 2026. La segmentación base distingue 32.017 usuarios free puros, 516 pagadores activos y 103 usuarios churned. Para el análisis de límites se trabaja a nivel de eventos de uso y ventanas de 30 días. En estado free se incluyen tanto usuarios que nunca han pagado como usuarios que posteriormente pagaron, siempre que el evento analizado ocurriera antes de la fecha de suscripción.

**Periodo temporal utilizado:**  
El análisis utiliza los datos disponibles hasta el 7 de febrero de 2026. Las preguntas y tests cubren el histórico disponible de la plataforma. El chat y los costes asociados dependen del periodo de trazabilidad disponible en `traces` y `ai_spans`. Las flashcards presentan una limitación temporal específica, ya que fueron lanzadas el 3 de enero de 2026 y, por tanto, solo disponen de aproximadamente cinco semanas de observación.

**Tablas de origen utilizadas:**  
- `users.csv`: información de usuarios, estado de suscripción, fecha de registro, fecha de suscripción y `stripe_customer_id`.
- `tests.csv`: tests creados, usuario creador y fecha de creación.
- `questions.csv`: preguntas generadas, incluyendo la variable `ai_assisted`.
- `notes.csv`: notas de flashcards, usuario creador, fecha y variable `ai_assisted`.
- `cards.csv`: tarjetas generadas a partir de notas.
- `traces.csv`: trazas de uso de funcionalidades de IA.
- `ai_spans.csv`: llamadas individuales a modelos de IA, coste, estado y modelo.
- `stripe_subscriptions.csv`: información de suscripciones, precios y fecha de creación.
- `buys.csv`: compras de extras, aunque no constituye el bloque central de resultados de esta versión.

**Variables y métricas generadas:**  
- Segmento de usuario: `free_puro`, `pagador_activo` y `churned`.
- Indicador de evento en estado free.
- Ventanas de 30 días desde la fecha de registro para eventos en estado free.
- Ventanas de 30 días desde la fecha de suscripción para eventos posteriores a la suscripción.
- Número de preguntas IA por usuario y ventana.
- Número de flashcards IA por usuario y ventana.
- Coste de chat por usuario y ventana.
- Indicador de superación de límite por funcionalidad.
- Usuarios que superan límites en estado free.
- Usuarios que superan límites en MagIA y SuperMagIA.
- Usuarios MagIA que tocan techo y posible conversión posterior a SuperMagIA.
- Heavy users por funcionalidad y solapamiento entre uso intensivo de preguntas, flashcards y chat.

**Resultados principales:**  
- En estado free, 17.136 usuarios generaron preguntas con IA y 611 superaron el límite de 250 preguntas en alguna ventana de 30 días, lo que representa el 3,57%.
- De los usuarios que superaron el límite de preguntas en estado free, 27 tuvieron relación con Stripe alguna vez y 17 son pagadores activos en la fecha de corte.
- En flashcards, 5.599 usuarios generaron flashcards en estado free y 201 superaron el límite de 500 flashcards, lo que representa el 3,59%.
- De los usuarios que superaron el límite de flashcards, 10 tuvieron relación con Stripe y los 10 son pagadores activos en la fecha de corte.
- En chat, 2.410 usuarios utilizaron el chat en estado free y 82 superaron el presupuesto gratuito de 0,05 dólares, lo que representa el 3,40%.
- De los usuarios que superaron el presupuesto de chat, 19 tuvieron relación con Stripe y 17 son pagadores activos.
- En el plan MagIA, los límites se superan en una proporción reducida: 9 usuarios superan el límite de preguntas, 1 supera el límite de flashcards y 1 supera el presupuesto de chat.
- En el plan SuperMagIA, ningún usuario supera los límites de preguntas, flashcards o chat en las ventanas analizadas.
- Ninguno de los 9 usuarios MagIA que superaron el límite de preguntas tiene registrada una suscripción posterior a SuperMagIA.
- El análisis de solapamiento muestra que la mayoría de heavy users lo son en una sola funcionalidad: el 85,2% de los usuarios intensivos destaca únicamente en una feature, frente al 12,3% que destaca en dos y el 2,4% que destaca en las tres.

**Figuras o tablas que alimentan el TFG:**  
- Tabla de superación de límites en estado free por funcionalidad.
- Tabla de superación de límites en MagIA.
- Tabla de superación de límites en SuperMagIA.
- Resultado de conversión MagIA → SuperMagIA tras tocar techo.
- Tabla o resumen de heavy users y solapamiento entre funcionalidades.
- Correlación de uso entre preguntas, flashcards y chat.

**Relación con otros notebooks:**  
Este notebook complementa al Notebook 01, que fija la base general de usuarios y monetización, y al Notebook 02, que identifica patrones de conversión inmediata y conversión por uso. El Notebook 03 profundiza en si la conversión por uso puede estar asociada al agotamiento de límites. Los resultados muestran que los límites afectan a una minoría de usuarios en estado free, por lo que no explican de forma agregada la mayor parte de la conversión. Sin embargo, la superación de límites permite identificar usuarios intensivos, especialmente en chat, donde la proporción de usuarios con relación posterior o actual con Stripe es mayor que en preguntas y flashcards.

**Limitaciones metodológicas:**  
- El análisis es descriptivo y no permite establecer causalidad entre superación de límites y conversión.
- El estado free se reconstruye temporalmente a partir de `stripe_customer_id`, `subscription_created_at` y la fecha del evento, por lo que puede existir cierta aproximación.
- La validación con Stripe muestra que `subscription_created_at` coincide con la primera suscripción en la mayoría de casos, aunque existen algunos usuarios con diferencias superiores a 24 horas.
- El análisis de chat depende de la trazabilidad disponible en `traces` y `ai_spans`.
- El análisis de flashcards está limitado por el lanzamiento reciente de esta funcionalidad.
- La identificación de planes MagIA y SuperMagIA se basa en precios de Stripe y puede simplificar casos de cambios de plan, descuentos o prorrateos.
- La superación del límite indica fricción o uso intensivo, pero no prueba que el límite haya causado la conversión.

**Conclusión operativa para el TFG:**  
Este notebook debe mantenerse como bloque principal de análisis de límites y fricción freemium. Sus resultados muestran que los límites actuales no generan fricción masiva, ya que solo alrededor del 3,4%-3,6% de los usuarios con uso en estado free supera los límites de preguntas, flashcards o chat. No obstante, la fricción no tiene el mismo valor económico en todas las funcionalidades: el chat presenta una asociación más fuerte con monetización que preguntas y flashcards. En los planes de pago, los límites parecen ser suficientes para la mayoría de usuarios, especialmente en SuperMagIA, donde ningún usuario alcanza los techos cuantitativos. Esto sugiere que las recomendaciones de negocio no deberían centrarse solo en endurecer límites, sino en identificar perfiles intensivos por funcionalidad y diseñar estrategias diferenciadas de conversión, retención y upselling.

In [25]:
# ============================================================
# EXPORTAR USUARIOS QUE SUPERAN ALGÚN LÍMITE EN ESTADO FREE
# ============================================================

# Ajusta estos nombres si en tu notebook se llaman distinto
df_preguntas_limites = preguntas_free_ventana.copy()
df_flashcards_limites = flashcards_free_ventana.copy()
df_chat_limites = chat_free_ventana.copy()

# Asegurar IDs como string
df_preguntas_limites["user_id"] = df_preguntas_limites["user_id"].astype(str).str.strip()
df_flashcards_limites["user_id"] = df_flashcards_limites["user_id"].astype(str).str.strip()
df_chat_limites["user_id"] = df_chat_limites["user_id"].astype(str).str.strip()

# Usuarios que superan límite de preguntas
usuarios_superan_preguntas = set(
    df_preguntas_limites.loc[
        df_preguntas_limites["supera_limite"] == True,
        "user_id"
    ]
)

# Usuarios que superan límite de flashcards
usuarios_superan_flashcards = set(
    df_flashcards_limites.loc[
        df_flashcards_limites["supera_limite"] == True,
        "user_id"
    ]
)

# Usuarios que superan límite de chat
usuarios_superan_chat = set(
    df_chat_limites.loc[
        df_chat_limites["supera_limite"] == True,
        "user_id"
    ]
)

# Unión: usuarios que superan al menos un límite
usuarios_superan_algún_limite = (
    usuarios_superan_preguntas
    | usuarios_superan_flashcards
    | usuarios_superan_chat
)

# Tabla final a exportar
usuarios_superan_limites_pre = pd.DataFrame({
    "user_id": list(usuarios_superan_algún_limite)
})

usuarios_superan_limites_pre["supera_limite_pre"] = 1

usuarios_superan_limites_pre["supera_limite_preguntas_pre"] = (
    usuarios_superan_limites_pre["user_id"].isin(usuarios_superan_preguntas)
).astype(int)

usuarios_superan_limites_pre["supera_limite_flashcards_pre"] = (
    usuarios_superan_limites_pre["user_id"].isin(usuarios_superan_flashcards)
).astype(int)

usuarios_superan_limites_pre["supera_limite_chat_pre"] = (
    usuarios_superan_limites_pre["user_id"].isin(usuarios_superan_chat)
).astype(int)

print("Usuarios que superan límite de preguntas:", len(usuarios_superan_preguntas))
print("Usuarios que superan límite de flashcards:", len(usuarios_superan_flashcards))
print("Usuarios que superan límite de chat:", len(usuarios_superan_chat))
print("Usuarios que superan al menos un límite:", len(usuarios_superan_algún_limite))

usuarios_superan_limites_pre.head()

Usuarios que superan límite de preguntas: 611
Usuarios que superan límite de flashcards: 201
Usuarios que superan límite de chat: 82
Usuarios que superan al menos un límite: 863


,user_id,supera_limite_pre,supera_limite_preguntas_pre,supera_limite_flashcards_pre,supera_limite_chat_pre
0,019b89c4-02f6-707a-beb5-b73444036401,1,1,0,0
1,edeea97a-e17f-7170-898c-5889ca0100aa,1,1,0,0
2,019b9fdb-d9c8-7e17-8cd0-037b60f129c1,1,1,0,0
3,019bc2a9-21c2-7bfa-9740-d68ea722569d,1,1,1,0
4,019bc049-2a51-7689-b53d-93aae2ed7496,1,1,0,0


In [26]:
# Guardar para usar en Notebook 04
usuarios_superan_limites_pre.to_csv(
    "../outputs/usuarios_superan_limites_pre.csv",
    index=False
)

print("✅ Guardado en: usuarios_superan_limites_pre.csv")

✅ Guardado en: usuarios_superan_limites_pre.csv


In [28]:
# ============================================================
# Buscar DataFrames que tengan user_id + ventana
# ============================================================

dfs = []

for name, obj in list(globals().items()):
    if isinstance(obj, pd.DataFrame):
        cols_lower = [str(c).lower() for c in obj.columns]
        tiene_user = "user_id" in cols_lower
        tiene_ventana = any("ventana" in c for c in cols_lower)
        
        if tiene_user and tiene_ventana:
            dfs.append({
                "nombre": name,
                "filas": len(obj),
                "columnas": obj.columns.tolist()
            })

for d in dfs:
    print("\n" + "="*80)
    print("DataFrame:", d["nombre"])
    print("Filas:", d["filas"])
    print("Columnas:")
    print(d["columnas"])


DataFrame: preguntas_estado_free
Filas: 1631316
Columnas:
['id', 'content', 'explication', 'ai_assisted', 'is_updated', 'new_version_id', 'is_random', 'test_id', 'options', 'version', 'created_at', 'updated_at', 'deleted_at', 'ai_assisted_bool', 'user_id', 'test_created_at', 'user_created_at', 'subscription_created_at', 'stripe_customer_id', 'era_free_en_ese_momento', 'dias_desde_registro', 'ventana_30d']

DataFrame: preguntas_free_ventana
Filas: 17496
Columnas:
['user_id', 'ventana_30d', 'preguntas_ia', 'supera_limite']

DataFrame: cards_estado_free
Filas: 757759
Columnas:
['id', 'note_id', 'ord', 'created_at', 'updated_at', 'deleted_at', 'user_id', 'note_created_at', 'user_created_at', 'subscription_created_at', 'stripe_customer_id', 'era_free_en_ese_momento', 'dias_desde_registro', 'ventana_30d']

DataFrame: flashcards_free_ventana
Filas: 5607
Columnas:
['user_id', 'ventana_30d', 'cards_generadas', 'supera_limite']

DataFrame: chat_estado_free
Filas: 19881
Columnas:
['trace_id', 'u

In [29]:
from pathlib import Path

OUTPUT_PATH = Path("outputs")
OUTPUT_PATH.mkdir(exist_ok=True)

df_preguntas_limites.to_csv(OUTPUT_PATH / "df_preguntas_limites.csv", index=False)
df_flashcards_limites.to_csv(OUTPUT_PATH / "df_flashcards_limites.csv", index=False)
df_chat_limites.to_csv(OUTPUT_PATH / "df_chat_limites.csv", index=False)

print("Tablas de límites exportadas correctamente.")

Tablas de límites exportadas correctamente.
